In [2]:
from pyscf import gto, scf, fci
import netket as nk
import netket.experimental as nkx
import jax.numpy as jnp
from flax import nnx
import jax

# 设置H4分子的几何构型
# 使用线性构型，H-H键长为1.0埃
bond_length = 1.0
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
    ('H', (2*bond_length, 0., 0.)),
    ('H', (3*bond_length, 0., 0.))
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')
mf = scf.RHF(mol).run(verbose=0)

# 进行FCI计算作为参考
E_fci = fci.FCI(mf).kernel()[0]
print(f"FCI能量: {E_fci:.8f} Ha")
ham_nk = nkx.operator.from_pyscf_molecule(mol)
hilbert_space = ham_nk.hilbert

E0 = float(nk.exact.lanczos_ed(ham_nk))
print(f"NetKet精确对角化能量: {E0:.8f} Ha")


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FCI能量: -2.16638745 Ha
NetKet精确对角化能量: -2.16638745 Ha


/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_73420/1707035674.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  E0 = float(nk.exact.lanczos_ed(ham_nk))


In [14]:
from netket.nn.masked_linear import default_kernel_init
from functools import partial
class LogNeuralJastrowSlater(nnx.Module):
    hilbert: nk.hilbert.SpinOrbitalFermions

    def __init__(
        self,
        hilbert,
        jastrow_hidden_units: int,
        kernel_init=default_kernel_init,
        param_dtype=float,
        *,
        rngs: nnx.Rngs,
    ):
        self.hilbert = hilbert

        # To generate random numbers we need to extract the key from the `rngs` object.
        key = rngs.params()

        # the N x Nf matrix of the orbitals
        self.M = nnx.Param(
            kernel_init(
                key,
                (
                    self.hilbert.n_orbitals,
                    self.hilbert.n_fermions,
                ),
                param_dtype,
            )
        )

        # The Neural Jastrow wf
        self.jastrow_linear = nnx.Linear(
            in_features=hilbert.n_orbitals,
            out_features=jastrow_hidden_units,
            param_dtype=param_dtype,
            rngs=rngs,
        )

    def __call__(self, n: jax.Array) -> jax.Array:
        # For simplicity, we write a function that operates on a single configuration of size (N,)
        # and we vectorize it using `jnp.vectorize` with the signature='(n)->()' argument, which specifies
        # that the function is defined to operate on arrays of shape (n,) and return scalars.
        @partial(jnp.vectorize, signature="(n)->()")
        def log_sd(n):
            # Construct the Neural Jastrow
            J = self.jastrow_linear(n)
            J = jax.nn.tanh(J)
            J = J.sum()

            # Construct the slater determinant
            R = n.nonzero(size=self.hilbert.n_fermions)[0]
            # Extract from the (N, Nf) matrix the (Nf, Nf) submatrix of M corresponding to the occupied orbitals.
            A = self.M[R]
            log_slater = _logdet_cmplx(A)

            # Multiply log slater with the jastrow (and since we are working in log space, this becomes a sum)
            return log_slater + J

        return log_sd(n)

In [15]:
model = LogNeuralJastrowSlater(hilbert_space, jastrow_hidden_units=8, rngs=nnx.Rngs(2))

# Define a Metropolis exchange sampler
sa = nk.sampler.MetropolisLocal(hilbert_space)

# Define an optimizer
op = nk.optimizer.Sgd(learning_rate=0.05)

# Create a variational state
vstate = nk.vqs.MCState(sa, model, n_samples=2**12)

# Create a Variational Monte Carlo driver
preconditioner = nk.optimizer.SR(diag_shift=0.05)
gs = nk.VMC(H, op, variational_state=vstate, preconditioner=preconditioner)
# Construct the logger to visualize the data later on
nj_log = nk.logging.RuntimeLog()

# Run the optimization for 300 iterations
gs.run(n_iter=300, out=nj_log)

TypeError: dot_general requires contracting dimensions to have the same shape, got (8,) and (4,).

In [22]:
nnx.display(model)